---

**Analytical Validation Tests for 2D FEM Solver**

- Poiseuille Flow: in-plane shear, no-slip at walls
- Couette Flow: in-plane shear, wall velocity
- Bernoulli Venturi: inertia terms
---

## Poiseuille Flow

Pressure-driven flow in $x$-direction with no-slip walls as $y$-boundary condition.

**Setup:**
- Body force $f_x$ drives flow in $x$-direction
- Periodic boundary conditions in $x$
- No-slip walls at $y=0$ and $y=L_y$: $j_x = j_y = 0$
- In-plane viscous diffusion terms active
- Wall stress terms $\tau_{xz}, \tau_{yz}$ deactivated

**Analytical solution:**

$$u(y) = \frac{\rho \cdot f_x}{2\mu} \cdot y \cdot (L_y - y)$$

The Dirichlet BC is applied on ghost cells, so the effective zero-velocity boundary is at the cell faces $y=0$ and $y=L_y$.

$\rightarrow$ **[YAML File](../../tests/configs/poiseuille_2d_body_force.yaml)**

In [ ]:
from GaPFlow.problem import Problem

problem = Problem.from_yaml("../../tests/configs/poiseuille_2d_body_force.yaml")
problem.q[0][:] = 1.0  # Uniform density
problem.run()

Compare with analytical solution:

In [ ]:
from utils.plotting import plot_poiseuille_validation

plot_poiseuille_validation(problem)

## Couette Flow

Shear-driven flow with stationary bottom wall and moving top wall.

**Setup:**
- Top wall moving with velocity $U$ (implemented as Dirichlet BC: $j_x = \rho \cdot U$)
- Bottom wall stationary: $j_x = j_y = 0$
- Periodic boundary conditions in $x$
- In-plane viscous diffusion terms active

**Analytical solution:**

$$u(y) = U \cdot \frac{y}{L_y}$$

The Dirichlet BC is applied via a mirror formula on ghost cells, so the effective boundary is at the cell faces $y=0$ (stationary) and $y=L_y$ (moving).

$\rightarrow$ **[YAML File](../../tests/configs/couette_2d_wall_velocity.yaml)**

In [ ]:
from GaPFlow.problem import Problem

problem_couette = Problem.from_yaml("../../tests/configs/couette_2d_wall_velocity.yaml")
problem_couette.q[0][:] = 1.0  # Uniform density
problem_couette.run()

Compare with analytical solution:

In [ ]:
from utils.plotting import plot_couette_validation

plot_couette_validation(problem_couette)

## Bernoulli Venturi Nozzle

Inviscid flow through a symmetric venturi channel, validating convective momentum terms.

<img src="./figures/Geometry_BernoulliVenturi.png"
     alt="Bernoulli venturi geometry"
     width="250">

**Setup:**
- Symmetric venturi: $h_{\mathrm{inlet}} = 1\,\mathrm{mm} \to h_{\mathrm{throat}} = 0.5\,\mathrm{mm} \to h_{\mathrm{outlet}} = 1\,\mathrm{mm}$
- DH EOS (nearly incompressible)
- Inlet: prescribed mass flux $j_x = \rho \cdot v = 5000\,\mathrm{kg/(m^2 \cdot s)}$
- Outlet: prescribed density $\rho = 1000\,\mathrm{kg/m^3}$
- Inertia terms active
- Custom topography via `topo.set_mapped_height()`

**Analytical predictions (Bernoulli equation):**

assuming constant density:

$$v \cdot h = \mathrm{const} \quad \Rightarrow \quad v_{\mathrm{throat}} = v_{\mathrm{inlet}} \cdot \frac{h_{\mathrm{inlet}}}{h_{\mathrm{throat}}} = 2 \cdot v_{\mathrm{inlet}}$$

$$p + \frac{1}{2}\rho v^2 = \mathrm{const} \quad \Rightarrow \quad \Delta p = \frac{1}{2}\rho(v_{\mathrm{throat}}^2 - v_{\mathrm{inlet}}^2)$$

$\rightarrow$ **[YAML File](../../tests/configs/bernoulli_venturi.yaml)**

In [ ]:
from GaPFlow.problem import Problem
from utils.topography import regen_bernoulli_venturi
from utils.plotting import plot_bernoulli_validation

RHO0, V_INLET = 1000.0, 5.0

problem = Problem.from_yaml("../../tests/configs/bernoulli_venturi.yaml")
regen_bernoulli_venturi(problem)
problem.q[1][:] = RHO0 * V_INLET  # Set inlet momentum as initial guess
problem.run()


Compare with Bernoulli predictions:

In [ ]:
plot_bernoulli_validation(problem)